# Interactive viewer: ATERA WTA FFPE breast cancer

`08_spatial_atera_breast_cancer_example.ipynb` ends with 170k annotated cells: cell types, CellCharter niches, a UMAP, and the morphology image with per-cell boundary polygons.
This notebook renders that result as a linked [Vitessce](http://vitessce.io/) viewer inside the notebook.

The viewer supports:

- Pan/zoom over the full-resolution morphology image with every cell boundary drawn on top.
- Recoloring cells by cell type or spatial niche, and toggling populations on and off.
- Selecting a gene, which recolors tissue and UMAP and updates a per-cell-type distribution.
- Per-cell tooltips, and selections made in the UMAP shown in the tissue.

Vitessce reads OME-NGFF images, segmentations and an AnnData table directly from a `SpatialData` Zarr store and renders through WebGL, so the work here is writing that store and declaring the views and the parameters they share.

**Prerequisites**

- `08_spatial_atera_breast_cancer_example.ipynb` run to its final cell, which writes `atera_annotated.h5ad`.
- The raw `outs/` bundle from notebook 08; the morphology image and cell boundaries are re-read from it.
- `vitessce` and its widget dependencies (in `requirements.txt`).
- A browser with internet access: the widget loads the Vitessce JS bundle from `unpkg.com`.

In [1]:
from pathlib import Path

import anndata as ad
import numpy as np
import spatialdata as sd
import spatialdata_io as sio
from spatialdata.models import TableModel
from vitessce import (
    CoordinationLevel as CL,
    CoordinationType as ct,
    SpatialDataWrapper,
    ViewType as vt,
    VitessceConfig,
    get_initial_coordination_scope_prefix,
    hconcat,
    vconcat,
)

ad.settings.allow_write_nullable_strings = False
ad.settings.auto_shard_zarr_v3 = False


In [2]:
DATA_ROOT = Path("./data/atera")
OUTS_DIR = DATA_ROOT / "outs"
ANNOTATED_H5AD = DATA_ROOT / "atera_annotated.h5ad"
VIEWER_ZARR = DATA_ROOT / "atera_viewer.zarr"

IMAGE_ELEMENT = "morphology_focus"
SHAPES_ELEMENT = "cell_boundaries"
VIEWER_TABLE = "viewer"
DATASET_UID = "A"

DEFAULT_GENE = "ESR1"
VIEWER_OBS = [
    "region",
    "cell_id",
    "cell_type",
    "niche_cellcharter",
    "total_counts",
    "n_genes_by_counts",
    "cell_area",
    "pct_control",
]

## 1. Load the annotated result of notebook 08

In [3]:
for path in (ANNOTATED_H5AD, OUTS_DIR):
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing -- run 08_spatial_atera_breast_cancer_example.ipynb first.")

adata = ad.read_h5ad(ANNOTATED_H5AD)
adata

AnnData object with n_obs × n_vars = 168303 × 18028
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'nucleus_to_cell_area', 'control_total', 'pct_control', 'transcripts_per_area', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'leiden_expr', 'cell_type', 'niche_cellcharter'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'total_counts', 'mean_counts', 'pct_dropout_by_counts', 'log1p_total_counts', 'log1p_mean_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'cell_type_co_occurrence', 'cell_type_colors', 'cell_type_nhood_enrichment', 'hvg', 'leiden_expr', 'leiden_expr_colors', 'log1p', 'neighbors', 'niche_cellcharter_colors', 'pca', 'spatial_neighbors

## 2. Assemble a browser-ready store

Vitessce streams from a Zarr store, so this section writes one holding what the viewer needs:

- `images/morphology_focus` and `shapes/cell_boundaries`, re-read from the `outs/` bundle and written as a multiscale OME-NGFF pyramid.
- `tables/viewer`, with the full log-normalized matrix, the `obs` columns, and the UMAP from notebook 08.

The matrix is stored as CSC.
Vitessce reads a single column per selected gene from a CSC matrix, so all 18,028 genes stay searchable without the browser ever holding the full matrix; a dense matrix or CSR would have to be scanned in full.

`spatialdata_io.xenium` loads the cell table even with `cells_table=False` (spatialdata-io 0.7.1), so the unused raw table is dropped before writing to keep the store lean.

Writing the image pyramid takes a few minutes and ~8 GB of disk, so it is skipped once the store exists.

In [4]:
if not VIEWER_ZARR.exists():
    raw = sio.xenium(
        OUTS_DIR,
        cells_boundaries=True,
        nucleus_boundaries=False,
        cells_as_circles=False,
        cells_labels=False,
        nucleus_labels=False,
        transcripts=False,
        morphology_mip=False,
        morphology_focus=True,
        aligned_images=False,
        cells_table=False,
    )
    del raw.tables["table"]
    raw.write(VIEWER_ZARR)

sdata = sd.read_zarr(VIEWER_ZARR)
sdata

SpatialData object, with associated Zarr store: /raid/lheumos/atera/atera_viewer.zarr
├── Images
│     └── 'morphology_focus': DataTree[cyx] (4, 28048, 46543), (4, 14024, 23271), (4, 7012, 11635), (4, 3506, 5817), (4, 1753, 2908)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (168303, 1) (2D shapes)
└── Tables
      └── 'viewer': AnnData (168303, 18028)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_boundaries (Shapes)

The panel matrix goes into `layers["lognorm"]`, and `X` is left empty deliberately.
`spatialdata` <= 0.7.3 validates a table's keys through `AnnData.layers`, and `anndata` >= 0.13 exposes `X` there under a `None` key, so writing any table with a populated `X` raises `AttributeError: 'NoneType' object has no attribute 'lower'`.
A named layer avoids that, and Vitessce reads the layer directly; `spatialdata` 0.8.0 fixes the underlying check.

`anndata.settings.allow_write_nullable_strings = False` in the imports cell is the second half of the same story.
Under pandas 3 string columns carry the `str` dtype, which anndata writes as a `nullable-string-array` group (`values` + `mask`); Vitessce's SpatialData table loader reads the var index as a plain array and fails with `Expected a zarr array at tables/viewer/var/_index, but found a group`.
The setting restores the older non-nullable encoding for every string column.

The cell-boundary polygons are subset to the cells that survived QC in notebook 08.
Vitessce pairs the polygons with the table's `obs` index by position, so leaving all 170,057 polygons against 168,303 table rows shifts every colour after the first dropped cell.

Sharding is disabled for the same reason.
anndata shards zarr v3 arrays by default, and a sharded array can only be read through a store that implements `getRange`; plain chunks keep the table readable through the simplest store interface.

In [5]:
viewer = ad.AnnData(
    obs=adata.obs[VIEWER_OBS].copy(),
    var=adata.var[[]].copy(),
    layers={"lognorm": adata.layers["lognorm"].tocsc()},
    obsm={"X_umap": np.asarray(adata.obsm["X_umap"], dtype=np.float32)},
)
viewer

AnnData object with n_obs × n_vars = 168303 × 18028
    obs: 'region', 'cell_id', 'cell_type', 'niche_cellcharter', 'total_counts', 'n_genes_by_counts', 'cell_area', 'pct_control'
    obsm: 'X_umap'
    layers: 'lognorm'

In [6]:
sdata.shapes[SHAPES_ELEMENT] = sdata[SHAPES_ELEMENT].loc[viewer.obs_names]
sdata.tables[VIEWER_TABLE] = TableModel.parse(
    viewer, region=SHAPES_ELEMENT, region_key="region", instance_key="cell_id"
)

on_disk = {path.rsplit("/", 1)[-1] for path in sdata.elements_paths_on_disk()}
for element in (SHAPES_ELEMENT, VIEWER_TABLE):
    if element in on_disk:
        sdata.delete_element_from_disk(element)
    sdata.write_element(element)
sdata

SpatialData object, with associated Zarr store: /raid/lheumos/atera/atera_viewer.zarr
├── Images
│     └── 'morphology_focus': DataTree[cyx] (4, 28048, 46543), (4, 14024, 23271), (4, 7012, 11635), (4, 3506, 5817), (4, 1753, 2908)
├── Shapes
│     └── 'cell_boundaries': GeoDataFrame shape: (168303, 1) (2D shapes)
└── Tables
      └── 'viewer': AnnData (168303, 18028)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_boundaries (Shapes)

## 3. Configure the viewer

A Vitessce config is a dataset, a set of views, and a declaration of which parameters those views share.

The morphology image has four channels.
DAPI is shown first with a contrast window taken from the coarsest pyramid level; the others can be switched on in the layer controller.

In [7]:
image = sdata[IMAGE_ELEMENT]
channels = list(sd.models.get_channel_names(image))
dapi_thumbnail = image[list(image)[-1]]["image"].sel(c="DAPI").to_numpy()
dapi_max = float(np.percentile(dapi_thumbnail, 99.5))

channels, dapi_max

(['DAPI', 'ATP1A1/CD45/E-Cadherin', '18S', 'alphaSMA/Vimentin'], 4325.0)

In [8]:
vc = VitessceConfig(schema_version="1.0.18", name="ATERA WTA FFPE breast cancer")

wrapper = SpatialDataWrapper(
    sdata_store=str(VIEWER_ZARR),
    image_path=f"images/{IMAGE_ELEMENT}",
    obs_segmentations_path=f"shapes/{SHAPES_ELEMENT}",
    table_path=f"tables/{VIEWER_TABLE}",
    obs_feature_matrix_path=f"tables/{VIEWER_TABLE}/layers/lognorm",
    obs_set_paths=[
        f"tables/{VIEWER_TABLE}/obs/cell_type",
        f"tables/{VIEWER_TABLE}/obs/niche_cellcharter",
    ],
    obs_set_names=["Cell type", "Spatial niche"],
    obs_embedding_paths=[f"tables/{VIEWER_TABLE}/obsm/X_umap"],
    obs_embedding_names=["UMAP"],
    region=SHAPES_ELEMENT,
    coordinate_system="global",
    coordination_values={"obsType": "cell"},
)
dataset = vc.add_dataset(name="ATERA breast cancer", uid=DATASET_UID).add_object(wrapper)

spatial = vc.add_view("spatialBeta", dataset=dataset)
layer_controller = vc.add_view("layerControllerBeta", dataset=dataset)
umap = vc.add_view(vt.SCATTERPLOT, dataset=dataset, mapping="UMAP")
obs_sets = vc.add_view(vt.OBS_SETS, dataset=dataset)
feature_list = vc.add_view(vt.FEATURE_LIST, dataset=dataset)
distribution = vc.add_view(vt.OBS_SET_FEATURE_VALUE_DISTRIBUTION, dataset=dataset)

Views are linked through shared coordination scopes.
The selected gene, the color encoding, the colormap and the selected cell sets live in scopes that the tissue, UMAP, gene list and distribution plot all read, so a selection in one updates the rest.

In [9]:
color_encoding, feature_selection, colormap, set_selection = vc.add_coordination(
    ct.OBS_COLOR_ENCODING,
    ct.FEATURE_SELECTION,
    ct.FEATURE_VALUE_COLORMAP,
    ct.OBS_SET_SELECTION,
)
color_encoding.set_value("cellSetSelection")
feature_selection.set_value([DEFAULT_GENE])
colormap.set_value("plasma")
set_selection.set_value([["Cell type", c] for c in viewer.obs["cell_type"].cat.categories])

for view in (umap, obs_sets, feature_list, distribution):
    view.use_coordination(color_encoding, feature_selection, colormap, set_selection)

vc.link_views(
    [spatial, layer_controller, umap, obs_sets, feature_list, distribution], [ct.OBS_TYPE], ["cell"]
)

vc.link_views_by_dict(
    [spatial, layer_controller],
    {
        "imageLayer": CL([{
            "photometricInterpretation": "BlackIsZero",
            "imageChannel": CL([{
                "spatialTargetC": channels.index("DAPI"),
                "spatialChannelColor": [255, 255, 255],
                "spatialChannelWindow": [0.0, dapi_max],
            }]),
        }]),
    },
    scope_prefix=get_initial_coordination_scope_prefix(DATASET_UID, "image"),
)

vc.link_views_by_dict(
    [spatial, layer_controller],
    {
        "segmentationLayer": CL([{
            "spatialLayerVisible": True,
            "spatialLayerOpacity": 1.0,
            "segmentationChannel": CL([{
                "obsType": "cell",
                "spatialChannelVisible": True,
                "spatialChannelOpacity": 0.7,
                "spatialSegmentationFilled": True,
                "spatialSegmentationStrokeWidth": 0.5,
                "obsColorEncoding": color_encoding,
                "featureSelection": feature_selection,
                "featureValueColormap": colormap,
                "obsSetSelection": set_selection,
                "tooltipsVisible": True,
            }]),
        }]),
    },
    scope_prefix=get_initial_coordination_scope_prefix(DATASET_UID, "obsSegmentations"),
)

vc.layout(
    vconcat(
        hconcat(spatial, vconcat(umap, distribution), split=[2, 1]),
        hconcat(layer_controller, obs_sets, feature_list),
        split=[2, 1],
    )
)

## 4. Explore

Passing `sdata_store` rather than `sdata_path` registers the Zarr store with the widget, so chunks are pulled over the kernel comm channel instead of an HTTP server on localhost.
The data stays on this machine, and no port needs forwarding when the kernel is remote (VS Code Remote-SSH, JupyterHub).

The Vitessce JS bundle itself is still fetched from `unpkg.com` by the browser.

In [10]:
vw = vc.widget(height=900)
vw

In the viewer:

- Zoom into a duct until single boundaries resolve, then lower the segmentation opacity in the layer controller to compare labels against the stain.
- Switch the cell-set tree from *Cell type* to *Spatial niche* for the CellCharter domains.
- Search `KIT`, then `ANKRD30A` in the gene list (all 18,028 are searchable): the two DCIS substates of notebook 08 separate in place.
- Enable the `ATP1A1/CD45/E-Cadherin` channel to check segmentation against the membrane stain.
- Deselect everything except `Tumor_Hypoxic` and the immune populations.

`vc.export(to="files", base_url=..., out_dir=...)` writes a static config and data bundle for hosting elsewhere, and `vc.to_dict(base_url=...)` returns the underlying JSON.